# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors.

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on.

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you.


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly?
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised).

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

In [ ]:
from datetime import datetime
import copy

import datasets
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ds_labeled = datasets.load_dataset("google-research-datasets/paws", "labeled_final")
ds = datasets.load_dataset("google-research-datasets/paws", "unlabeled_final")

labeled_final/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 8.43MB            

labeled_final/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

labeled_final/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 1.24MB            

labeled_final/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

labeled_final/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.23MB            

...

unlabeled_final/train-00000-of-00001.par(…): downloading bytes:           |  0.00B            

unlabeled_final/validation-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 1.66MB            

unlabeled_final/validation-00000-of-0000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/645652 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [5]:
def train_tokenizer(corpus, vocab_size, min_frequency, pretokenizer=Metaspace(), unk_token="[UNK]"):
    """
    corpus: a list of texts (each a string)
    vocab_size: number of distinct BPE tokens
    min_frequency: we only merge token pairs that are frequent enough
    pretokenizer: a simple tokenizer that we use before BPE tokenization

        You probably don't need to change the pretokenizer, unless you are working with
        languages that really don't use any form of blank space
        (then you could, for example, pretokenize at character level)
    """
    # Create an empty BPE tokenizer
    tokenizer = Tokenizer(BPE(unk_token=unk_token)) #unk_token used for characters that cannot be encoded

    # First split the corpus into words
    tokenizer.pre_tokenizer = pretokenizer

    # Tell BPE how large the vocabulary should be
    trainer = BpeTrainer(
        vocab_size=vocab_size, # The size of the vocabulary to learn.
        min_frequency=min_frequency, # Only perform a merge if the pair occurs at least min_frequency times in the training corpus.
        special_tokens=[unk_token]
    )

    # Learn the vocabulary and merge rules
    tokenizer.train_from_iterator(corpus, trainer=trainer)

    return tokenizer

def tokenize_batch(batch):
    return {
        "ids1": [
            tokenizer.encode(s).ids
            for s in batch["sentence1"]
        ],
        "ids2": [
            tokenizer.encode(s).ids
            for s in batch["sentence2"]
        ],
    }


In [16]:
VOCAB_SIZE = 2048
MAX_LEN = 256
BS = 16

tokenizer = train_tokenizer(
    corpus=list(ds['train']['sentence1']) + list(ds['train']['sentence2']),
    vocab_size=VOCAB_SIZE - 1,
    min_frequency=2,
)
tokenizer.add_special_tokens(["[PAD]"])
PAD_ID = tokenizer.token_to_id("[PAD]")

encoded = ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["id", "sentence1", "sentence2"],
)

encoded_labeled = ds_labeled.map(
    tokenize_batch,
    batched=True,
    remove_columns=["id", "sentence1", "sentence2"],
)

def collate_fn(batch):
    def pad(sequences):
        out = torch.full(
            (len(sequences), MAX_LEN),
            PAD_ID,
            dtype=torch.long,
        )
        for i, ids in enumerate(sequences):
            out[i, :len(ids)] = torch.tensor(ids)
        return out

    x1 = pad([item["ids1"] for item in batch])
    x2 = pad([item["ids2"] for item in batch])
    y = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    return x1, x2, y

train_loader = DataLoader(
    encoded["train"],
    batch_size=BS,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    encoded["validation"],
    batch_size=BS,
    collate_fn=collate_fn,
)

train_loader_labeled = DataLoader(
    encoded_labeled["train"],
    batch_size=BS,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader_labeled = DataLoader(
    encoded_labeled["validation"],
    batch_size=BS,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    encoded_labeled["test"],
    batch_size=BS,
    collate_fn=collate_fn,
)

In [ ]:
class DualLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        pad_id=PAD_ID,
        embed_dim=128,
        lstm_hidden_dim=128,
        lstm_layers=2,
        linear_hidden_dim=128,
        dropout=.2,
    ):
        super().__init__()
        self.pad_id = pad_id

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout)

        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden_dim,
            batch_first=True,
            num_layers=lstm_layers,
            bidirectional=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(2 * 4 * lstm_hidden_dim, linear_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(linear_hidden_dim, 2),
        )

    def _encode(self, seq):
        lengths = (seq != self.pad_id).sum(dim=1).cpu()

        emb = self.embedding(seq)
        emb = self.embedding_dropout(emb)

        # Avoid diminishing gradient through padding
        packed = nn.utils.rnn.pack_padded_sequence(
            emb,
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )

        _, (h_n, _) = self.encoder(packed)

        return torch.cat([h_n[-2], h_n[-1]], dim=-1) # Bidirectional

    def forward(self, seq_1, seq_2, inf=False):
        h_n_1 = self._encode(seq_1)
        h_n_2 = self._encode(seq_2)

        logits = self.classifier(
            torch.cat([
                h_n_1,
                h_n_2,
                torch.abs(h_n_1 - h_n_2),
                h_n_1 * h_n_2
            ], dim=-1)
        )

        return torch.softmax(logits, dim=-1) if inf == True else logits

In [8]:
model = DualLSTMClassifier()

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"{trainable_params} model params")

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


def run_epoch(loader, optimizer=None, training=True):
    model.train(training)

    total_loss = .0
    total_correct = 0
    total = 0

    for x1, x2, y in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)
        y = y.to(device)

        with torch.set_grad_enabled(training):
            logits = model(x1, x2)
            loss = criterion(logits, y)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, total_correct / total

1053058 model params


In [ ]:
MAX_EPOCHS = 30
PATIENCE = 3

best_val_loss = 999
best_weights = None
epochs_without_improvement = 0
prev_epoch_time = datetime.now()

for epoch in range(1, MAX_EPOCHS + 1):

    train_loss, train_acc = run_epoch(train_loader, optimizer, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"({datetime.now() - prev_epoch_time}) | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.3f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val acc: {val_acc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        break

    prev_epoch_time = datetime.now()

model.load_state_dict(best_weights)

Epoch 01 | (0:17:14.995634) | Train loss: 0.5433 | Train acc: 0.729 | Val loss: 0.5903 | Val acc: 0.756
Epoch 02 | (0:17:11.617304) | Train loss: 0.4528 | Train acc: 0.795 | Val loss: 0.5615 | Val acc: 0.786
Epoch 03 | (0:17:15.065904) | Train loss: 0.4032 | Train acc: 0.822 | Val loss: 0.4492 | Val acc: 0.827
Epoch 04 | (0:17:56.619006) | Train loss: 0.3701 | Train acc: 0.838 | Val loss: 0.4217 | Val acc: 0.840
Epoch 05 | (0:17:49.293607) | Train loss: 0.3464 | Train acc: 0.850 | Val loss: 0.3720 | Val acc: 0.855
Epoch 06 | (0:17:42.231074) | Train loss: 0.3270 | Train acc: 0.859 | Val loss: 0.3846 | Val acc: 0.855
Epoch 07 | (0:17:33.309105) | Train loss: 0.3119 | Train acc: 0.866 | Val loss: 0.3827 | Val acc: 0.855
Epoch 08 | (0:17:34.877757) | Train loss: 0.2976 | Train acc: 0.872 | Val loss: 0.3688 | Val acc: 0.860
Epoch 09 | (0:17:28.734253) | Train loss: 0.2847 | Train acc: 0.878 | Val loss: 0.3386 | Val acc: 0.875
Epoch 10 | (0:17:26.147344) | Train loss: 0.2731 | Train acc: 0.

<All keys matched successfully>

In [ ]:
torch.save({"wts": model.state_dict()}, "model_unlabeled.pth")

In [ ]:
model.load_state_dict(best_weights)

optimizer_ft = torch.optim.AdamW([
    {
        "params": model.embedding.parameters(),
        "lr": 1e-6
    },
    {
        "params": model.encoder.parameters(),
        "lr": 1e-5
    },
    {
        "params": model.classifier.parameters(),
        "lr": 1e-5
    }
])#, weight_decay=0.01)

MAX_EPOCHS_FT = 15
PATIENCE_FT = 3

best_val_loss_ft = 999
best_weights_ft = copy.deepcopy(best_weights)
epochs_without_improvement = 0
prev_epoch_time = datetime.now()

for epoch in range(1, MAX_EPOCHS_FT + 1):

    train_loss, train_acc = run_epoch(train_loader_labeled, optimizer_ft, training=True)
    val_loss, val_acc = run_epoch(val_loader_labeled, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"({datetime.now() - prev_epoch_time}) | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.3f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val acc: {val_acc:.3f}"
    )

    if val_loss < best_val_loss_ft:
        best_val_loss_ft = val_loss
        best_weights_ft = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE_FT:
        break

    prev_epoch_time = datetime.now()

model.load_state_dict(best_weights_ft)

Epoch 01 | (0:01:26.419378) | Train loss: 0.3986 | Train acc: 0.847 | Val loss: 0.4094 | Val acc: 0.849
Epoch 02 | (0:01:25.606872) | Train loss: 0.3643 | Train acc: 0.851 | Val loss: 0.4053 | Val acc: 0.849
Epoch 03 | (0:01:26.329239) | Train loss: 0.3503 | Train acc: 0.854 | Val loss: 0.4071 | Val acc: 0.848
Epoch 04 | (0:01:26.594380) | Train loss: 0.3396 | Train acc: 0.856 | Val loss: 0.4052 | Val acc: 0.849
Epoch 05 | (0:01:26.688551) | Train loss: 0.3296 | Train acc: 0.861 | Val loss: 0.4133 | Val acc: 0.848
Epoch 06 | (0:01:26.009738) | Train loss: 0.3214 | Train acc: 0.863 | Val loss: 0.4155 | Val acc: 0.849
Epoch 07 | (0:01:26.072822) | Train loss: 0.3151 | Train acc: 0.866 | Val loss: 0.4127 | Val acc: 0.849
Early stopping at epoch 7


<All keys matched successfully>

In [30]:
torch.save({"wts": model.state_dict()}, "model_ft_labeled.pth")

In [31]:
model.load_state_dict(best_weights)

test_loss, test_acc = run_epoch(test_loader, training=False)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.2%}")

Test loss: 0.5221
Test accuracy: 85.16%
